# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** Built on my ML-04 data contract and compared against my frozen ML-07 baseline.

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the same pages, windows and label as ML-04 and ML-07

- Data: warehouse partitions `month=2026-03` (main) and `month=2026-04` (time-forward check only),
  plus `dim_content` (`content_created_date` only). June 2026 stays sealed.
- One frame-builder, `build_frame(T)`, makes the ML-04 frame for any decision date T:
  - feature window: the 14 days before T (week 1 + week 2);
  - outcome window: the 14 days from T;
  - pages: ≥ 100 impressions before T, tracked from the window's first day, with a valid position.
- Main frame: **T = 2026-03-16**, exactly the ML-04/ML-07 frame (checked against their receipts).
- Later frame: **T = 2026-04-13**, used only for the time-forward check in section 3.
- Label `declined_next_2w`: outcome-window impressions more than 20% below the feature window.

In [1]:
import os, sys, subprocess, importlib.util, json, time
from pathlib import Path

if importlib.util.find_spec('duckdb') is None:      # Colab ships duckdb; this covers fresh local envs
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'duckdb'])

import duckdb
import numpy as np
import pandas as pd
import sklearn
from huggingface_hub import hf_hub_download, list_repo_files

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ')

REPO = 'FlyRank/internship-warehouse'
MONTHS = ['2026-03', '2026-04']     # development months; 2026-06 is the sealed test month

def fetch(path):
    # downloads once into the HF cache; later runs reuse the local copy
    return Path(hf_hub_download(REPO, path, repo_type='dataset', token=HF_TOKEN)).as_posix()

repo_files = list_repo_files(REPO, repo_type='dataset', token=HF_TOKEN)
daily_paths = [fetch(f) for f in repo_files
               if any(f.startswith(f'fact_content_daily_performance/month={m}/') for m in MONTHS)]
con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute(f"CREATE OR REPLACE VIEW daily AS SELECT * FROM read_parquet({daily_paths})")
con.execute(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{fetch('dim_content.parquet')}')")
sql = lambda q: con.sql(q).df()

KEYS, LABEL = ['client_hash_id', 'content_hash_id'], 'declined_next_2w'
created = sql('SELECT content_hash_id, content_created_date FROM dim_content')

def build_frame(T):
    T = pd.Timestamp(T)
    f_start, f_end, wk2 = T - pd.Timedelta(days=14), T - pd.Timedelta(days=1), T - pd.Timedelta(days=7)
    o_start, o_end = T, T + pd.Timedelta(days=13)
    d = lambda x: f"DATE '{x.date()}'"
    ok = 'gsc_data_available IS TRUE'
    pre = sql(f'''
        SELECT client_hash_id, content_hash_id,
               MIN(report_date)                                                      AS first_seen,
               SUM(gsc_impressions) FILTER (WHERE {ok})                              AS imp_pre,
               SUM(gsc_impressions) FILTER (WHERE {ok} AND report_date <  {d(wk2)})  AS imp_wk1,
               SUM(gsc_impressions) FILTER (WHERE {ok} AND report_date >= {d(wk2)})  AS imp_wk2,
               MAX(gsc_impressions) FILTER (WHERE {ok})                              AS max_day_pre,
               SUM(gsc_sum_position) FILTER (WHERE {ok} AND gsc_avg_position > 0)
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE {ok} AND gsc_avg_position > 0), 0) AS pos_pre,
               SUM(gsc_sum_position) FILTER (WHERE {ok} AND gsc_avg_position > 0 AND report_date < {d(wk2)})
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE {ok} AND gsc_avg_position > 0
                                                       AND report_date < {d(wk2)}), 0)  AS pos_wk1,
               SUM(gsc_sum_position) FILTER (WHERE {ok} AND gsc_avg_position > 0 AND report_date >= {d(wk2)})
                 / NULLIF(SUM(gsc_impressions) FILTER (WHERE {ok} AND gsc_avg_position > 0
                                                       AND report_date >= {d(wk2)}), 0) AS pos_wk2,
               COUNT(*) FILTER (WHERE {ok})                                          AS active_days_pre
        FROM daily
        WHERE report_date BETWEEN {d(f_start)} AND {d(f_end)}
        GROUP BY ALL
        HAVING imp_pre >= 100
    ''')
    out = sql(f'''
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) FILTER (WHERE {ok}) AS imp_out
        FROM daily
        WHERE report_date BETWEEN {d(o_start)} AND {d(o_end)}
        GROUP BY ALL
    ''')
    p = pre.merge(created, on='content_hash_id', how='left')
    p = p[pd.to_datetime(p['first_seen']) == f_start].dropna(subset=['pos_pre']).reset_index(drop=True)
    p[['imp_wk1', 'imp_wk2']] = p[['imp_wk1', 'imp_wk2']].fillna(0)
    # the five contract features (ML-04)
    p['log_imp_pre']   = np.log1p(p['imp_pre'])
    p['momentum_pre']  = np.log((p['imp_wk2'] + 1) / (p['imp_wk1'] + 1))
    p['age_days_at_T'] = (T - pd.to_datetime(p['content_created_date'])).dt.days
    # three extra pre-T candidates, tested in section 3
    p['peak_day_share_pre'] = p['max_day_pre'] / p['imp_pre']
    site = p.groupby('client_hash_id')[['imp_wk1', 'imp_wk2']].sum()
    p['site_momentum'] = p['client_hash_id'].map(np.log((site['imp_wk2'] + 1) / (site['imp_wk1'] + 1)))
    p['pos_delta'] = (p['pos_wk2'] - p['pos_wk1']).fillna(0)
    # outcome — label only
    p = p.merge(out, on=KEYS, how='left')
    p['imp_out'] = p['imp_out'].fillna(0)
    p[LABEL] = (p['imp_out'] < 0.8 * p['imp_pre']).astype(int)
    assert (p['age_days_at_T'] > 0).all()
    # DuckDB returns GROUP BY rows in a different order each run; a fixed order makes seeded models reproducible
    return p.sort_values(KEYS).reset_index(drop=True)

t0 = time.time()
mar, apr = build_frame('2026-03-16'), build_frame('2026-04-13')
print(f'March frame (T=2026-03-16): {len(mar):,} pages, {mar.client_hash_id.nunique()} clients, base rate {mar[LABEL].mean():.3f}')
print(f'April frame (T=2026-04-13): {len(apr):,} pages, {apr.client_hash_id.nunique()} clients, base rate {apr[LABEL].mean():.3f}')
print(f'built in {time.time() - t0:.0f}s | pandas {pd.__version__}, scikit-learn {sklearn.__version__}, duckdb {duckdb.__version__}')

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'work' / 'notebooks').is_dir()), Path.cwd())
OUT_DIR = ROOT / 'work' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
receipt04, receipt07 = OUT_DIR / 'w03_data_contract_receipts.json', OUT_DIR / 'w04_baseline_metrics.json'
if receipt04.exists():                             # continuity with ML-04, when run inside the repo
    frame04 = json.loads(receipt04.read_text())['frame']
    assert (frame04['pages'], frame04['base_rate']) == (len(mar), round(float(mar[LABEL].mean()), 3))
    print('March frame = the ML-04 contract frame (same pages, same base rate)')

March frame (T=2026-03-16): 74,542 pages, 36 clients, base rate 0.337
April frame (T=2026-04-13): 81,947 pages, 41 clients, base rate 0.417
built in 20s | pandas 2.3.3, scikit-learn 1.7.2, duckdb 1.5.6
March frame = the ML-04 contract frame (same pages, same base rate)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane asks **"which pages first?"**, and I have an observed yes/no label. The skill's table says:
take a classifier's probability and judge it as a ranking, at precision@K. So every candidate below
outputs a score, the queue is sorted by that score, and the main metric is **precision@50** (from
ML-03: about one editor-sprint of reviews).

| Candidate | From the session menu | Why it's on the list |
|---|---|---|
| Logistic Regression | yes | readable: one weight per feature, and its probabilities rank well |
| Decision Tree (depth 3) | yes | the most readable model there is; a test of whether a few if-rules are enough |
| Random Forest | yes | catches non-linear effects and interactions; robust defaults |
| Gradient Boosting (`HistGradientBoosting`) | yes, "where safe" | safe here: every feature is from before T, so a strong learner can't find a leak |
| **References, not models** | | |
| ML-07 rule (frozen) | — | the baseline to beat, recomputed in this run |
| Slide sort (no training) | — | ranks by `momentum_pre` alone: does *learning* add anything beyond my one CONFIRMED signal? |

**Features:** the five contract features from ML-04, each knowable at T. I also test three extra
pre-T candidates once, in cross-validation: `peak_day_share_pre` (the ML-07 spike signal),
`site_momentum` (whole-site slide) and `pos_delta` (position change from week 1 to week 2). They stay
only if they earn it.

**Selection rule, fixed before I look at any test client:** within the training clients, take the
best mean average precision (AP) in grouped CV, then choose the **simplest learned model within one
standard deviation of it**. Complexity has to earn its place. Default settings, no tuning,
`random_state = 42` everywhere.

In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
FEATURES = ['log_imp_pre', 'pos_pre', 'active_days_pre', 'momentum_pre', 'age_days_at_T']   # ML-04 contract
EXTRA = ['peak_day_share_pre', 'site_momentum', 'pos_delta']
OUTCOME_COLS = {'imp_out', LABEL}
assert not OUTCOME_COLS & set(FEATURES + EXTRA), 'no outcome-window column may be a feature'

# simplest first: this order is the tie-break in the selection rule
CANDIDATES = {
    'logistic_regression': (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), FEATURES),
    'decision_tree_d3':    (lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, random_state=SEED), FEATURES),
    'random_forest':       (lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED), FEATURES),
    'gradient_boosting':   (lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=SEED), FEATURES),
    'logistic_regression+3': (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), FEATURES + EXTRA),
    'random_forest+3':     (lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED), FEATURES + EXTRA),
}

def rule_score(df):
    # the frozen ML-07 rule: impressions lost when the week-over-week slide exceeds 20%, else 0;
    # imp_pre only breaks ties (always < 1, scores are whole impressions), exactly like the ML-07 queue order
    flagged = (df['imp_wk2'] + 1) / (df['imp_wk1'] + 1) < 0.8
    return np.where(flagged, df['imp_wk1'] - df['imp_wk2'], 0) + 0.5 * df['imp_pre'] / (df['imp_pre'].max() + 1)

REFERENCES = {'ML-07 rule (baseline)': rule_score,
              'slide sort (no training)': lambda df: -df['momentum_pre']}

def precision_at_k(y, score, k):
    return float(np.asarray(y)[np.argsort(-np.asarray(score), kind='stable')[:k]].mean())

def metrics(y, score):
    m = {f'P@{k}': precision_at_k(y, score, k) for k in (10, 20, 50, 100, 500)}
    m.update({'AP': average_precision_score(y, score), 'ROC-AUC': roc_auc_score(y, score)})
    return m

pd.DataFrame([(name, len(cols), ', '.join(cols[len(FEATURES):]) or '—') for name, (_, cols) in CANDIDATES.items()],
             columns=['candidate', 'n_features', 'extra features'])

,candidate,n_features,extra features
0,logistic_regression,5,—
1,decision_tree_d3,5,—
2,random_forest,5,—
3,gradient_boosting,5,—
4,logistic_regression+3,8,"peak_day_share_pre, site_momentum, pos_delta"
5,random_forest+3,8,"peak_day_share_pre, site_momentum, pos_delta"


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client.** A real deployment scores sites the model has never seen, and sites differ a lot.
In ML-04 the decline rate ranged from 0.10 to 0.74 between clients. A random row split would put
pages of the same site on both sides and let the model learn each site's habits. So:

1. **The comparison split = the exact ML-04 / ML-07 split.** `GroupShuffleSplit(test_size=0.3,
   random_state=42)` on `client_hash_id`: 25 training clients and 11 held-out clients. The baseline was
   scored on these same 11 clients in ML-07 (precision@50 = 0.60), so the comparison is like-for-like.
2. **Model selection happens only inside the training clients**, using `GroupKFold(5)` by client. The
   held-out clients are touched once, for the final table.

**Time-aware.** Features come only from before T and the label only from after T (ML-04 contract). On
top of that I run a **time-forward check**: train on the March frame (training clients, labels end
03-29), then test on the **April frame (T = 04-13, features start 03-30)**, held-out clients only.
That is a later period *and* unseen sites, with no overlap between training labels and test features.

In [3]:
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=SEED)
                           .split(mar, groups=mar['client_hash_id']))
train, test = mar.iloc[train_idx].reset_index(drop=True), mar.iloc[test_idx].reset_index(drop=True)
train_clients, test_clients = set(train['client_hash_id']), set(test['client_hash_id'])
apr_test = apr[apr['client_hash_id'].isin(test_clients)].reset_index(drop=True)

assert not train_clients & test_clients, 'a client may sit on one side only'
mar_label_end, apr_feature_start = pd.Timestamp('2026-03-29'), pd.Timestamp('2026-03-30')
assert mar_label_end < apr_feature_start, 'March training labels must end before April test features begin'

split = pd.DataFrame({
    'pages':     [len(train), len(test), len(apr_test)],
    'clients':   [len(train_clients), len(test_clients), apr_test['client_hash_id'].nunique()],
    'base rate': [train[LABEL].mean(), test[LABEL].mean(), apr_test[LABEL].mean()],
}, index=['train (March, 25 clients)', 'held-out test (March)', 'time-forward test (April, same held-out clients)']).round(3)
split

,pages,clients,base rate
"train (March, 25 clients)",37756,25,0.313
held-out test (March),36786,11,0.361
"time-forward test (April, same held-out clients)",37993,11,0.437


Train: 37,756 pages from 25 clients (base rate 0.313). Held-out: 36,786 pages from 11 clients (base rate 0.361). These are exactly the pages ML-07 was scored on. April check: 37,993 pages from the same 11 clients (base rate 0.437). No client sits on both sides, and the March training labels end (03-29) before the April test features begin (03-30).

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Step A — choose the model inside the training clients (grouped 5-fold CV)

In [4]:
rows = []
for fold, (fit_idx, val_idx) in enumerate(GroupKFold(n_splits=5).split(train, groups=train['client_hash_id'])):
    fit, val = train.iloc[fit_idx], train.iloc[val_idx]
    for name, scorer in REFERENCES.items():
        rows.append({'fold': fold, 'model': name, **metrics(val[LABEL], scorer(val))})
    for name, (make, cols) in CANDIDATES.items():
        score = make().fit(fit[cols], fit[LABEL]).predict_proba(val[cols])[:, 1]
        rows.append({'fold': fold, 'model': name, **metrics(val[LABEL], score)})
cv = pd.DataFrame(rows)

cv_table = cv.groupby('model', sort=False)[['P@50', 'P@500', 'AP', 'ROC-AUC']].agg(['mean', 'std']).round(3)
learned = cv_table.loc[list(CANDIDATES), ('AP', 'mean')]
best = learned.idxmax()
bar = learned[best] - cv_table.loc[best, ('AP', 'std')]
CHOSEN = next(name for name in CANDIDATES if learned[name] >= bar)     # simplest within one std of the best
print(f'best mean AP: {best} ({learned[best]:.3f}); bar = best - 1 std = {bar:.3f}  ->  chosen: {CHOSEN}\n')
cv_table

best mean AP: random_forest+3 (0.489); bar = best - 1 std = 0.331  ->  chosen: logistic_regression



P@50         P@500            AP        ROC-AUC  \
                           mean    std   mean    std   mean    std    mean   
model                                                                        
ML-07 rule (baseline)     0.368  0.127  0.402  0.206  0.382  0.151   0.584   
slide sort (no training)  0.892  0.088  0.584  0.253  0.475  0.166   0.649   
logistic_regression       0.892  0.074  0.615  0.192  0.486  0.156   0.660   
decision_tree_d3          0.772  0.073  0.620  0.164  0.427  0.125   0.626   
random_forest             0.860  0.058  0.582  0.240  0.476  0.161   0.647   
gradient_boosting         0.836  0.108  0.572  0.222  0.461  0.159   0.631   
logistic_regression+3     0.896  0.082  0.608  0.218  0.485  0.160   0.663   
random_forest+3           0.860  0.093  0.619  0.206  0.489  0.158   0.661   

                                 
                            std  
model                            
ML-07 rule (baseline)     0.074  
slide sort (no training)  0.082  
logistic_regression       0.072  
decision_tree_d3          0.061  
random_forest             0.084  
gradient_boosting         0.082  
logistic_regression+3     0.082  
random_forest+3           0.094

**Inside the training clients, the learned models are within noise of each other.** Mean AP runs from 0.427 (depth-3 tree) to 0.489 (random forest + 3 extra features), while the fold-to-fold spread is about 0.15. That spread is bigger than any gap between models. So the selection rule picks the simplest model that qualifies: **logistic regression on the five contract features** (mean AP 0.486).

The three extra features don't earn their place: +0.013 AP for the forest and −0.001 for logistic regression, both far inside the noise.

Two more things stand out. The frozen rule is the weakest (AP 0.382, P@50 0.368). And the slide sort, with no training at all, sits level with every learned model (AP 0.475, P@50 0.892).

### Step B — the comparison table on the held-out clients (the ML-07 split)

Every candidate is refit on all 25 training clients and scored once on the 11 held-out clients,
with the same label and the same metrics. The baseline is recomputed in this same run.

In [5]:
holdout_scores = {name: scorer(test) for name, scorer in REFERENCES.items()}
fitted = {}
for name, (make, cols) in CANDIDATES.items():
    fitted[name] = make().fit(train[cols], train[LABEL])
    holdout_scores[name] = fitted[name].predict_proba(test[cols])[:, 1]

holdout = pd.DataFrame({name: metrics(test[LABEL], s) for name, s in holdout_scores.items()}).T
holdout.loc['base rate (random order)'] = {**{f'P@{k}': test[LABEL].mean() for k in (10, 20, 50, 100, 500)},
                                           'AP': test[LABEL].mean(), 'ROC-AUC': 0.5}
holdout = holdout.round(3)

if receipt07.exists():                             # the recomputed baseline must match ML-07
    assert holdout.loc['ML-07 rule (baseline)', 'P@50'] == json.loads(receipt07.read_text())['hindsight']['ML-04 holdout clients']['precision@50']
    print('baseline reproduced: same precision@50 on the same held-out clients as ML-07\n')
holdout.loc[['base rate (random order)', 'ML-07 rule (baseline)', 'slide sort (no training)', *CANDIDATES]]

baseline reproduced: same precision@50 on the same held-out clients as ML-07



,P@10,P@20,P@50,P@100,P@500,AP,ROC-AUC
base rate (random order),0.361,0.361,0.361,0.361,0.361,0.361,0.500
ML-07 rule (baseline),0.800,0.650,0.600,0.570,0.560,0.515,0.677
slide sort (no training),1.000,1.000,0.980,0.960,0.864,0.587,0.706
logistic_regression,1.000,1.000,0.960,0.960,0.884,0.591,0.711
decision_tree_d3,0.900,0.850,0.880,0.890,0.670,0.505,0.648
random_forest,1.000,1.000,0.920,0.910,0.878,0.612,0.720
gradient_boosting,1.000,1.000,0.960,0.950,0.860,0.582,0.694
logistic_regression+3,1.000,1.000,0.980,0.930,0.868,0.588,0.721
random_forest+3,1.000,0.950,0.940,0.940,0.874,0.602,0.726


**The model beats the baseline on the agreed metric.** On the same 11 held-out clients, logistic regression reaches **precision@50 = 0.96, against 0.60 for the ML-07 rule** (base rate 0.361). Average precision is 0.591 vs 0.515, and ROC-AUC is 0.711 vs 0.677. The recomputed baseline matches ML-07 exactly.

**But complexity is not what wins:**
- The no-training slide sort gets P@50 0.98 and AP 0.587, the same as the model.
- The random forest has the best AP (0.612), but its P@50 is lower (0.92), and in cross-validation it was not better than logistic regression (AP 0.476 vs 0.486).
- The depth-3 tree is clearly worse (P@50 0.88, AP 0.505).

The win over the baseline comes from changing *what the queue is sorted by*: relative slide instead of absolute impressions lost. Section 4c shows what that change costs.

### Step C — time-forward check: train on March, test on April (unseen sites, later period)

In [6]:
forward = {name: scorer(apr_test) for name, scorer in REFERENCES.items()}
for name in ['logistic_regression', 'random_forest', 'gradient_boosting']:
    forward[name] = fitted[name].predict_proba(apr_test[CANDIDATES[name][1]])[:, 1]   # models trained on March only
forward = pd.DataFrame({name: metrics(apr_test[LABEL], s) for name, s in forward.items()}).T.round(3)
print(f"April held-out pages: {len(apr_test):,}, base rate {apr_test[LABEL].mean():.3f}\n")
forward

April held-out pages: 37,993, base rate 0.437



,P@10,P@20,P@50,P@100,P@500,AP,ROC-AUC
ML-07 rule (baseline),1.0,1.0,0.76,0.60,0.464,0.461,0.534
slide sort (no training),1.0,1.0,0.98,0.96,0.954,0.561,0.597
logistic_regression,1.0,1.0,0.94,0.96,0.930,0.526,0.559
random_forest,0.9,0.9,0.92,0.91,0.904,0.510,0.546
gradient_boosting,1.0,1.0,0.94,0.93,0.822,0.490,0.532


**The ordering holds in a later period on unseen sites.** In April, logistic regression (trained on March only) reaches P@50 = 0.94, against 0.76 for the rule (base rate 0.437). The plain slide sort does even better: P@50 0.98, and AP 0.561 against 0.526 for the model.

ROC-AUC falls to about 0.53–0.60 for every ranking. So below the top of the list the ranking is weak, and the trained models lose a little against the plain sort. Directional reading: what carries over to a new month is the slide signal itself, not anything finer the models learned from March.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### 4a — What the chosen model leans on
Standardised logistic weights, then permutation importance on the held-out clients: shuffle one
feature and measure how much AP drops.

In [7]:
from sklearn.inspection import permutation_importance

model, cols = fitted[CHOSEN], CANDIDATES[CHOSEN][1]
perm = permutation_importance(model, test[cols], test[LABEL], scoring='average_precision',
                              n_repeats=5, random_state=SEED, n_jobs=1)
lean = pd.DataFrame({'AP drop when shuffled': perm.importances_mean, 'spread (std)': perm.importances_std}, index=cols)
if hasattr(model, 'named_steps') and 'logisticregression' in model.named_steps:
    lean.insert(0, 'standardised weight', model.named_steps['logisticregression'].coef_[0])
lean.sort_values('AP drop when shuffled', ascending=False).round(3)

,standardised weight,AP drop when shuffled,spread (std)
momentum_pre,-0.756,0.205,0.002
age_days_at_T,-0.236,0.031,0.002
pos_pre,0.089,0.011,0.001
active_days_pre,0.003,0.000,0.000
log_imp_pre,-0.130,-0.004,0.001


**The model leans almost entirely on `momentum_pre`.** Shuffling it drops AP by 0.205. The next feature, `age_days_at_T`, costs only 0.031 and `pos_pre` 0.011; `active_days_pre` and `log_imp_pre` cost about nothing. The three that matter point the sensible way:

1. **`momentum_pre` (weight −0.76).** Pages already sliding before T keep sliding. This is the signal ML-07 CONFIRMED.
2. **`age_days_at_T` (−0.24).** Older pages are a little *less* likely to drop. That matches the ML-07 staleness check, where the oldest pages declined least: age works as a mild stability signal for a model, even though it failed as a refresh trigger.
3. **`pos_pre` (+0.09).** Pages further down the results are a little more likely to drop, since deeper positions are less stable.

Nothing is suspiciously perfect. The strongest feature has a plain explanation, and no feature comes from after T.

### 4b — Where it is wrong: by client, and three concrete cases

In [8]:
scored = test.assign(score=holdout_scores[CHOSEN])
scored['rank'] = scored['score'].rank(ascending=False, method='first').astype(int)

per_client = scored.groupby('client_hash_id').apply(
    lambda g: pd.Series({'pages': len(g), 'base_rate': g[LABEL].mean(),
                         'AP': average_precision_score(g[LABEL], g['score']) if g[LABEL].nunique() > 1 else np.nan}),
    include_groups=False)
per_client = per_client[per_client['pages'] >= 200].sort_values('pages', ascending=False)
per_client['AP_lift'] = per_client['AP'] / per_client['base_rate']
per_client.index = [f'site {i + 1}' for i in range(len(per_client))]   # IDs dropped: order by size is enough
print(f"held-out clients with >= 200 pages: {len(per_client)}; AP lift over base rate from "
      f"{per_client['AP_lift'].min():.2f}x to {per_client['AP_lift'].max():.2f}x\n")
print(per_client.round(3).to_string(), '\n')

show = ['rank', 'imp_wk1', 'imp_wk2', 'imp_out', 'momentum_pre', 'peak_day_share_pre', 'pos_pre', 'age_days_at_T', LABEL]
false_pos = scored[scored[LABEL] == 0].nsmallest(2, 'rank')            # ranked near the top, did not decline
false_neg = scored[scored[LABEL] == 1].nlargest(2, 'rank')             # declined, ranked near the bottom
print('Ranked near the top, did NOT decline:')
print(false_pos[show].round(2).to_string(index=False), '\n')
print('Declined, but ranked near the bottom:')
print(false_neg[show].round(2).to_string(index=False), '\n')

# the pattern behind the misses: a spike in week 2 makes a page look like it is rising
missed = scored[(scored[LABEL] == 1)]
low = missed[missed['rank'] > len(scored) * 0.9]
print(f"declined pages ranked in the bottom 10%: {len(low):,}; median peak-day share {low['peak_day_share_pre'].median():.2f} "
      f"vs {scored['peak_day_share_pre'].median():.2f} for all held-out pages")

held-out clients with >= 200 pages: 9; AP lift over base rate from 1.09x to 1.87x

          pages  base_rate     AP  AP_lift
site 1  13657.0      0.430  0.713    1.657
site 2  10494.0      0.445  0.484    1.089
site 3   7364.0      0.099  0.166    1.682
site 4   1662.0      0.295  0.553    1.872
site 5   1033.0      0.373  0.518    1.391
site 6    923.0      0.390  0.444    1.138
site 7    618.0      0.597  0.776    1.299
site 8    510.0      0.490  0.634    1.294
site 9    481.0      0.297  0.529    1.778 

Ranked near the top, did NOT decline:
 rank  imp_wk1  imp_wk2  imp_out  momentum_pre  peak_day_share_pre  pos_pre  age_days_at_T  declined_next_2w
   29    118.0      0.0    153.0         -4.78                0.41     8.43            360                 0
   40    103.0      2.0    283.0         -3.55                0.29    48.53             27                 0 

Declined, but ranked near the bottom:
 rank  imp_wk1  imp_wk2  imp_out  momentum_pre  peak_day_share_pre  pos_pre  age

**The lift varies a lot by site.** Across the 9 held-out sites with ≥ 200 pages, the model's AP is between 1.09× and 1.87× the site's own base rate. The second-biggest site (10,494 pages) gets almost no lift (1.09×), so this model is not a uniform win across sites.

**Three concrete wrong cases:**
1. **Ranked 29th, did not decline.** 118 impressions in week 1, 0 in week 2, then 153 in the two weeks after T. The page sits right on the 100-impression floor, where a zero week is noise, not decay. It bounced back.
2. **Ranked 40th, did not decline.** 103 → 2 → 283, at position 48.5. Same story: a tiny page deep in the results whose quiet week was random.
3. **Declined, but ranked 36,757th (near the bottom).** 0 impressions in week 1, then 536 in week 2, on a 20-day-old page. A brand-new page getting its first exposure looks like a huge rise (`momentum_pre` +6.29), and then it settled lower. The other miss shown (0 → 233 → 21) is the same pattern on an older page.

**The pattern behind the misses is the ML-07 spike problem in reverse:** a jump in week 2 makes a page look like it is rising. The 473 declined pages ranked in the bottom 10% have a median peak-day share of 0.20, against 0.13 for all held-out pages.

### 4c — Precision versus stake
The rule ranks by *impressions lost*, the model by *probability of decline*. How much search
visibility does each top 50 actually cover?

In [9]:
def stake(df, score, k=50):
    top = df.assign(s=np.asarray(score)).nlargest(k, 's')
    return {'precision@50': top[LABEL].mean(),
            'impressions before T': int(top['imp_pre'].sum()),
            'impressions actually lost after T': int((top['imp_pre'] - top['imp_out']).clip(lower=0).sum()),
            'median impressions per page': float(top['imp_pre'].median())}

p_mar, p_apr = holdout_scores[CHOSEN], fitted[CHOSEN].predict_proba(apr_test[cols])[:, 1]
POLICIES = {
    'ML-07 rule (baseline)':                  lambda df, p: rule_score(df),
    CHOSEN:                                   lambda df, p: p,
    f'{CHOSEN} x impressions (value-aware)':  lambda df, p: p * df['imp_pre'],   # expected impressions at risk
}
stake_table = pd.DataFrame({f'{period} | {name}': stake(df, policy(df, p))
                            for period, df, p in [('March', test, p_mar), ('April', apr_test, p_apr)]
                            for name, policy in POLICIES.items()}).T
stake_table

,precision@50,impressions before T,impressions actually lost after T,median impressions per page
March | ML-07 rule (baseline),0.60,1185414.0,382930.0,19342.0
March | logistic_regression,0.96,18040.0,16416.0,217.0
March | logistic_regression x impressions (value-aware),0.66,2653061.0,810794.0,44315.0
April | ML-07 rule (baseline),0.76,1254199.0,694126.0,21372.0
April | logistic_regression,0.94,125762.0,114239.0,393.0
April | logistic_regression x impressions (value-aware),0.62,2787998.0,833645.0,39097.5


**This is the most important error in the notebook: the model's precision win comes from tiny pages.** Its top 50 have a median of **217 impressions** per page, against 19,342 for the rule's top 50. After T, the model's top 50 lost **16,416** impressions in total, while the rule's lost **382,930**: 23× more visibility. Precision@50 counts a 100-impression page the same as a 50,000-impression page, so a model judged on it learns to put collapsed micro-pages first. An editor's hour is worth more on the big ones.

Multiplying the model's probability by the page's impressions ("expected impressions at risk") gives a **value-aware middle ground**:
- **March:** precision@50 0.66 (rule: 0.60) and 810,794 impressions lost, 2.1× the rule's top 50.
- **April:** precision@50 0.62 (rule: 0.76) and 833,645 impressions lost, 1.2× the rule's.

It beats the rule on stake in both months but on precision only in March, so it is a lead for the capstone, not a result. The real lesson is about the metric: the capstone must report precision@K *together with* impressions covered (or precision above a volume floor), never precision@50 alone.

### 4d — What did the model actually learn? A stricter label
My label compares the two weeks after T with the two weeks before. A page that collapsed in week 2 and
simply *stays* down counts as "declined". So I ask a harder question, for pages with ≥ 50
impressions in week 2: **did it fall *further*, more than 20% below its own week-2 level?**
(`fell_further` = weekly impressions after T < 0.8 × week-2 impressions.) It is evaluation only and is
never used as a feature.

In [10]:
def fell_further(df):
    return ((df['imp_out'] / 2) < 0.8 * df['imp_wk2']).astype(int)

hard_train, hard_test = train[train['imp_wk2'] >= 50], test[test['imp_wk2'] >= 50]
y_hard = fell_further(hard_test)
hard = {name: metrics(y_hard, scorer(hard_test)) for name, scorer in REFERENCES.items()}
hard[f'{CHOSEN} (trained on the contract label)'] = metrics(y_hard, fitted[CHOSEN].predict_proba(hard_test[cols])[:, 1])
for name in ['logistic_regression', 'random_forest']:
    m = CANDIDATES[name][0]().fit(hard_train[FEATURES], fell_further(hard_train))
    hard[f'{name} (retrained on fell_further)'] = metrics(y_hard, m.predict_proba(hard_test[FEATURES])[:, 1])
hard = pd.DataFrame(hard).T[['P@50', 'P@500', 'AP', 'ROC-AUC']].round(3)
print(f'pages: {len(hard_test):,}; base rate of fell_further: {y_hard.mean():.3f}\n')
print(hard.to_string(), '\n')

# is the random forest's edge stable? grouped 5-fold CV over the training clients
rf_folds = []
for fit_idx, val_idx in GroupKFold(n_splits=5).split(hard_train, groups=hard_train['client_hash_id']):
    fit, val = hard_train.iloc[fit_idx], hard_train.iloc[val_idx]
    s = CANDIDATES['random_forest'][0]().fit(fit[FEATURES], fell_further(fit)).predict_proba(val[FEATURES])[:, 1]
    rf_folds.append({'P@50': precision_at_k(fell_further(val), s, 50), 'base': fell_further(val).mean()})
rf_folds = pd.DataFrame(rf_folds)
print(f"random forest on fell_further, grouped CV inside training clients: P@50 from {rf_folds['P@50'].min():.2f} "
      f"to {rf_folds['P@50'].max():.2f} (fold base rates {rf_folds['base'].min():.2f}-{rf_folds['base'].max():.2f})")

pages: 35,524; base rate of fell_further: 0.334

                                                     P@50  P@500     AP  ROC-AUC
ML-07 rule (baseline)                                0.14  0.228  0.346    0.539
slide sort (no training)                             0.38  0.360  0.349    0.513
logistic_regression (trained on the contract label)  0.46  0.430  0.373    0.545
logistic_regression (retrained on fell_further)      0.20  0.278  0.392    0.595
random_forest (retrained on fell_further)            0.84  0.806  0.444    0.579 



random forest on fell_further, grouped CV inside training clients: P@50 from 0.36 to 0.68 (fold base rates 0.18-0.37)


**What the model learned is persistence, not fresh decline.** On the stricter `fell_further` question (35,524 pages, base rate 0.334):
- The slide sort drops to near chance: P@50 0.38, ROC-AUC 0.513.
- The rule falls *below* chance, at 0.14: its spike-driven picks mostly bounce back.
- The contract-trained model reaches only 0.46.

So most of the main win is pages that were already sliding and stayed down. That is still useful for a review queue, because it surfaces pages losing visibility right now. But it is not an early warning.

A random forest retrained on `fell_further` scores P@50 = 0.84 on the held-out clients. Inside the training clients, though, it swings from 0.36 to 0.68 across folds, so it is unstable. It is a capstone lead ("can anything predict a *new* drop?"), not a claim.

### Final table — model vs baseline (same 11 held-out clients, same label, same run)

In [11]:
FINAL_ROWS = {
    'base rate (random order)': lambda df, p: np.random.RandomState(SEED).rand(len(df)),
    'ML-07 rule (baseline)':    POLICIES['ML-07 rule (baseline)'],
    'slide sort (no training)': lambda df, p: -df['momentum_pre'],
    f'{CHOSEN} (chosen model)': POLICIES[CHOSEN],
    f'{CHOSEN} x impressions (value-aware lead)': POLICIES[f'{CHOSEN} x impressions (value-aware)'],
}
rows = []
for name, policy in FINAL_ROWS.items():
    s_mar, s_apr = policy(test, p_mar), policy(apr_test, p_apr)
    m, a = metrics(test[LABEL], s_mar), metrics(apr_test[LABEL], s_apr)
    rows.append({'ranking': name, 'P@10': m['P@10'], 'P@50': m['P@50'], 'P@500': m['P@500'],
                 'AP': m['AP'], 'ROC-AUC': m['ROC-AUC'],
                 'top-50 impr. lost': stake(test, s_mar)['impressions actually lost after T'],
                 'April P@50': a['P@50'], 'April AP': a['AP'],
                 'April top-50 impr. lost': stake(apr_test, s_apr)['impressions actually lost after T']})
final = pd.DataFrame(rows).set_index('ranking').round(3)
assert final.loc['ML-07 rule (baseline)', 'P@50'] == holdout.loc['ML-07 rule (baseline)', 'P@50']

receipts = {
    'notebook': 'work/notebooks/w05_model.ipynb', 'release': 'FlyRank/internship-warehouse v20260703',
    'frames': {'main_T': '2026-03-16', 'time_forward_T': '2026-04-13'},
    'split': 'GroupShuffleSplit(test_size=0.3, random_state=42) on client_hash_id (= ML-04/ML-07 split)',
    'selection': {'rule': 'simplest learned model within 1 std of best grouped-CV AP', 'best': best, 'chosen': CHOSEN},
    'features': cols, 'seed': SEED,
    'cv_inside_train': cv_table.round(3).set_axis([f'{m} {s}' for m, s in cv_table.columns], axis=1).to_dict(orient='index'),
    'holdout': holdout.to_dict(orient='index'), 'time_forward_april': forward.to_dict(orient='index'),
    'stake_top50': stake_table.to_dict(orient='index'), 'fell_further_check': hard.to_dict(orient='index'),
    'final_table': final.to_dict(orient='index'),
    'versions': {'pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'duckdb': duckdb.__version__},
}
(OUT_DIR / 'w05_model_metrics.json').write_text(json.dumps(receipts, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print('wrote', (OUT_DIR / 'w05_model_metrics.json').relative_to(ROOT), '\n')
final

wrote work\outputs\w05_model_metrics.json 



,P@10,P@50,P@500,AP,ROC-AUC,top-50 impr. lost,April P@50,April AP,April top-50 impr. lost
ranking,,,,,,,,,
base rate (random order),0.3,0.36,0.338,0.359,0.495,10779,0.48,0.440,18660
ML-07 rule (baseline),0.8,0.60,0.560,0.515,0.677,382930,0.76,0.461,694126
slide sort (no training),1.0,0.98,0.864,0.587,0.706,20107,0.98,0.561,152500
logistic_regression (chosen model),1.0,0.96,0.884,0.591,0.711,16416,0.94,0.526,114239
logistic_regression x impressions (value-aware lead),0.6,0.66,0.612,0.457,0.608,810794,0.62,0.399,833645


**What the errors look like:**
- **Who wins:** the chosen model (logistic regression on the five contract features) beats the frozen ML-07 baseline on the agreed metric and the same split: **precision@50 = 0.96 vs 0.60** on the 11 held-out clients. It still wins in a later month on the same unseen sites (0.94 vs 0.76).
- **Why it wins:** the win is one signal, the week-over-week slide. A sort by that signal, with no training, matches the model, so I do not claim the learned model adds much beyond it.
- **Where it is wrong:**
  - At the top, it chases tiny pages near the volume floor whose quiet week was noise.
  - Lower down, it misses pages whose week-2 jump (a launch or a spike) made them look like risers.
  - Its top 50 covers 23× less lost visibility than the rule's.
- **What it cannot do:** predict a *fresh* drop. It only catches the continuation of a drop already underway.
- **For next week's validation:** score precision *and* stake together, use a spike-robust measure, and report per-site results instead of one pooled number.

All numbers come from one seeded run (`random_state = 42`; pandas 2.3.3, scikit-learn 1.7.2). Tree-ensemble numbers can shift a little between library versions. The results are observational and directional, for decision support.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.